# Code Generator (Trình sinh mã)

Yêu cầu: dùng Frontier model (mô hình hàng đầu) để sinh mã C++ hiệu năng cao từ mã Python


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Nhắc nhở: lấy code mới nhất</h2>
            <span style="color:#f71;">Mình liên tục cải thiện các lab (bài thực hành), thêm ví dụ và bài tập.
            Đầu mỗi tuần, nên kiểm tra bạn đã có code mới nhất chưa.<br/>
            Trước hết hãy <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull và merge (gộp) thay đổi của bạn nếu cần</a>. Gặp vấn đề? Hỏi ChatGPT cách merge — hoặc liên hệ mình!<br/><br/>
            Sau khi pull code, từ thư mục llm_engineering, trong Cursor Terminal, chạy:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Lưu ý quan trọng</h1>
            <span style="color:#900;">
            Trong lab (bài thực hành) này, mình dùng các model cao cấp GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4 — những model có giá hơi cao hơn. Chi phí vẫn thấp, nhưng nếu bạn muốn giữ chi phí cực thấp, hãy chọn model rẻ hơn như gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Ô này nạp các thư viện cần dùng cho cả notebook.

import os  # Đọc biến môi trường (environment variable), ví dụ API Key.
from dotenv import load_dotenv  # Nạp khóa từ file .env vào môi trường.
from openai import OpenAI  # Client gọi Chat Completions API (cũng dùng cho Anthropic/Gemini/Grok).
import subprocess  # Chạy lệnh ngoài Python: compiler (trình biên dịch) và file thực thi.
from IPython.display import Markdown, display  # Hiện câu trả lời của model dạng markdown trong Jupyter.

In [ ]:
load_dotenv(override=True)  # Đọc .env; override=True: giá trị trong file ghi đè biến môi trường cũ.

openai_api_key = os.getenv('OPENAI_API_KEY')  # Lấy khóa OpenAI; None nếu chưa có.
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')  # Khóa Anthropic (Claude), tùy chọn.
google_api_key = os.getenv('GOOGLE_API_KEY')  # Khóa Google (Gemini), tùy chọn.
grok_api_key = os.getenv('GROK_API_KEY')  # Khóa Grok (xAI), tùy chọn.

if openai_api_key:  # Có khóa thì xác nhận đã nạp, chỉ in vài ký tự đầu (không lộ hết khóa).
    print(f"OpenAI API Key tồn tại và bắt đầu bằng {openai_api_key[:8]}")
else:
    print("Chưa thiết lập OpenAI API Key")  # Thiếu khóa OpenAI thì các ô gọi GPT sẽ lỗi.

if anthropic_api_key:
    print(f"Anthropic API Key tồn tại và bắt đầu bằng {anthropic_api_key[:7]}")
else:
    print("Chưa thiết lập Anthropic API Key (và đây là tùy chọn)")

if google_api_key:
    print(f"Google API Key tồn tại và bắt đầu bằng {google_api_key[:2]}")
else:
    print("Chưa thiết lập Google API Key (và đây là tùy chọn)")

if grok_api_key:
    print(f"Grok API Key tồn tại và bắt đầu bằng {grok_api_key[:4]}")
else:
    print("Chưa thiết lập Grok API Key (và đây là tùy chọn)")

In [ ]:
# Tạo các client: cùng class OpenAI, khác base_url (địa chỉ API) và khóa.

openai = OpenAI()  # Client OpenAI mặc định: dùng OPENAI_API_KEY, gọi api.openai.com.

anthropic_url = "https://api.anthropic.com/v1/"  # Endpoint Claude kiểu OpenAI-compatible.
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"  # Endpoint Gemini kiểu OpenAI.
grok_url = "https://api.x.ai/v1"  # Endpoint Grok (xAI).

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)  # Gọi Claude qua SDK OpenAI.
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)  # Gọi Gemini qua SDK OpenAI.
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)  # Gọi Grok qua SDK OpenAI.

In [ ]:
OPENAI_MODEL = "gpt-5"  # Tên model OpenAI dùng khi port (chuyển) code.
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"  # Tên model Claude.
GROK_MODEL = "grok-4"  # Tên model Grok.
GEMINI_MODEL = "gemini-2.5-pro"  # Tên model Gemini.

# Muốn giữ chi phí cực thấp? Bỏ dấu # ở các dòng dưới để ghi đè model rẻ hơn.

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-haiku-4-5"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-3.1-flash-lite"

## LƯU Ý:

Chúng ta sẽ viết giải pháp chuyển Python thành mã C++ hiệu quả, đã tối ưu (optimized) cho máy của bạn, rồi biên dịch thành native machine code (mã máy gốc) và chạy.

Bạn không bắt buộc phải tự chạy code — đó không phải mục tiêu chính của bài tập!

Nhưng nếu muốn (vì khá thú vị!), mình ghi các bước ở đây. Hoàn toàn tùy chọn!

Ngoài ra, mình cũng sẽ chỉ một website để bạn chạy mã C++.

In [ ]:
from system_info import retrieve_system_info  # Hàm trong repo: thu thập OS, CPU, compiler đã cài, ...

system_info = retrieve_system_info()  # Gọi hàm, lưu báo cáo máy thành chuỗi.
system_info  # Hiện báo cáo ở output ô Jupyter (giá trị cuối ô được in).

In [ ]:
# Prompt (lời nhắc) nhờ GPT xem máy đã có C++ compiler chưa và nên dùng lệnh compile/run nào.
message = f"""
Đây là báo cáo thông tin hệ thống (system information) của máy tính tôi.
Tôi muốn chạy C++ compiler (trình biên dịch C++) để biên dịch một file C++ tên main.cpp rồi thực thi theo cách đơn giản nhất.
Hãy trả lời xem tôi có cần cài C++ compiler nào không. Nếu có, hãy đưa hướng dẫn từng bước đơn giản nhất.

Nếu máy tôi đã sẵn sàng biên dịch C++, tôi muốn chạy đoạn Python tương tự như sau để biên dịch và thực thi:
```python
compile_command = # điền lệnh ở đây — để đạt runtime performance (hiệu năng khi chạy) nhanh nhất có thể
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # điền lệnh ở đây
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Hãy cho tôi chính xác nên dùng gì cho compile_command và run_command.

Thông tin hệ thống:
{system_info}
"""

# Gửi prompt tới GPT; role "user" = tin nhắn người dùng.
response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
# Lấy nội dung lựa chọn đầu và hiện markdown (hướng dẫn + lệnh) trong notebook.
display(Markdown(response.choices[0].message.content))
    

## Nếu bạn cần cài thêm phần mềm

Nếu muốn, hãy làm theo hướng dẫn của GPT! Sau đó chạy lại phần phân tích (có thể cần Restart notebook) để xác nhận đã sẵn sàng.

Bạn sẽ có lệnh biên dịch code và lệnh chạy chương trình!

Điền các lệnh đó vào ô bên dưới:

In [ ]:
# Lệnh biên dịch dạng list (danh sách đối số), an toàn hơn một chuỗi shell.
# Đây là lệnh mẫu trên máy Mac của giảng viên — Windows/Linux có thể khác; hãy thay bằng lệnh GPT gợi ý.
compile_command = [
    "clang++",              # Trình biên dịch C++ (Clang).
    "-std=c++17",           # Dùng chuẩn C++17.
    "-Ofast",               # Tối ưu tốc độ mạnh khi chạy (runtime).
    "-mcpu=native",         # Tối ưu đúng CPU máy này.
    "-flto=thin",           # LTO (tối ưu lúc liên kết) kiểu thin: nhanh hơn fat LTO.
    "-fvisibility=hidden",  # Ẩn symbol, binary gọn hơn.
    "-DNDEBUG",             # Tắt assert (kiểm tra debug) để chạy nhanh hơn.
    "main.cpp",             # File nguồn đầu vào.
    "-o", "main",           # Tên file thực thi đầu ra: main.
]
run_command = ["./main"]  # Chạy binary vừa tạo (kiểu Unix). Windows thường dùng ["main.exe"].

## Tiếp theo: nhiệm vụ chính

In [ ]:
# Luật cho model (system prompt): chỉ ra C++, output giống Python, càng nhanh càng tốt.
system_prompt = """
Nhiệm vụ của bạn là chuyển mã Python thành mã C++ hiệu năng cao (high performance).
Chỉ trả lời bằng mã C++. Không giải thích, trừ một vài comment (chú thích) khi cần.
Mã C++ phải cho ra output (kết quả in ra) giống hệt, trong thời gian ngắn nhất có thể.
"""

def user_prompt_for(python):
    # Tạo prompt user: nhét mã Python + thông tin máy + lệnh compile để model viết C++ phù hợp.
    return f"""
Port (chuyển) mã Python này sang C++ với implementation (cách hiện thực) nhanh nhất, cho ra output giống hệt trong thời gian ngắn nhất.
Thông tin hệ thống là:
{system_info}
Phản hồi của bạn sẽ được ghi vào file tên main.cpp rồi biên dịch và thực thi; lệnh compilation (biên dịch) là:
{compile_command}
Chỉ trả lời bằng mã C++.
Mã Python cần port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    # Đóng gói đúng format chat API: system = luật, user = bài toán.
    return [
        {"role": "system", "content": system_prompt},  # Vai trò hệ thống: ràng buộc cách trả lời.
        {"role": "user", "content": user_prompt_for(python)}  # Vai trò người dùng: mã Python cần chuyển.
    ]
 

In [ ]:
def write_output(cpp):
    # Ghi đè file main.cpp bằng mã C++ vừa nhận từ model.
    with open("main.cpp", "w", encoding="utf-8") as f:  # utf-8: comment tiếng Việt không lỗi encoding.
        f.write(cpp)  # Ghi toàn bộ chuỗi C++ vào file; with tự đóng file.

In [ ]:
def port(client, model, python):
    # Một lần chuyển Python → C++ rồi lưu file, dùng client + tên model được truyền vào.
    reasoning_effort = "high" if 'gpt' in model else None  # GPT reasoning: suy luận mạnh; model khác thì bỏ tham số này.
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)  # Gọi API sinh C++.
    reply = response.choices[0].message.content  # Lấy text lựa chọn đầu (thường còn markdown ```cpp).
    reply = reply.replace('```cpp','').replace('```','')  # Gỡ fence markdown để file .cpp compile được.
    write_output(reply)  # Ghi C++ sạch vào main.cpp.

In [ ]:
# Chuỗi mã Python dùng làm bài test: xấp xỉ π bằng vòng lặp rất nhiều lần (cố ý chậm).
# Đây là dữ liệu gửi cho model / exec, chưa chạy ngay khi định nghĩa.
pi = """
import time  # Thư viện đo thời gian thực thi.

def calculate(iterations, param1, param2):  # Tính chuỗi số; sau này nhân 4 sẽ ra xấp xỉ π.
    result = 1.0  # Điểm bắt đầu (số thực).
    for i in range(1, iterations+1):  # Lặp từ 1 đến iterations (200 triệu lần khi gọi bên dưới).
        j = i * param1 - param2  # Mẫu số thứ nhất của số hạng.
        result -= (1/j)  # Trừ 1/j.
        j = i * param1 + param2  # Mẫu số thứ hai.
        result += (1/j)  # Cộng 1/j.
    return result  # Trả tổng chuỗi.

start_time = time.time()  # Mốc bắt đầu đo.
result = calculate(200_000_000, 4, 1) * 4  # 200 triệu vòng; nhân 4 để ra π.
end_time = time.time()  # Mốc kết thúc đo.

print(f"Kết quả (Result): {result:.12f}")  # In π xấp xỉ, 12 chữ số thập phân.
print(f"Thời gian thực thi (Execution Time): {(end_time - start_time):.6f} giây")  # In thời gian, 6 chữ số.
"""

In [ ]:
def run_python(code):
    # Chạy chuỗi mã Python như một chương trình nhỏ (baseline để so với C++).
    globals = {"__builtins__": __builtins__}  # Namespace hạn chế; vẫn cho print/range qua builtins.
    exec(code, globals)  # Thực thi chuỗi code với namespace đó.

In [ ]:
run_python(pi)  # Chạy bài π bằng Python để lấy thời gian baseline (tham chiếu).

In [ ]:
port(openai, OPENAI_MODEL, pi)  # Nhờ GPT-5 viết C++ tương đương bài π, ghi vào main.cpp.

# Biên dịch C++ và thực thi

Ô tiếp theo chứa lệnh biên dịch file C++ dựa trên hướng dẫn từ GPT.

Một lần nữa, bước này không bắt buộc nếu bạn không muốn!

HOẶC cách khác: học viên Sandeep K.G. gợi ý có thể chạy Python và C++ online để thử. Cảm ơn Sandeep!  
> Không phải so sánh chính xác tuyệt đối, nhưng vẫn thấy được chênh lệch performance (hiệu năng).  
> Ví dụ tại: https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
# Dùng lệnh compile/run đã khai báo ở trên (từ GPT hoặc lệnh mẫu).

def compile_and_run():
    # Biên dịch main.cpp rồi chạy binary 3 lần để giảm nhiễu đo thời gian.
    subprocess.run(compile_command, check=True, text=True, capture_output=True)  # check=True: lỗi compile thì dừng; text+capture: bắt stdout/stderr dạng str.
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)  # Lần chạy 1: in output (kết quả + thời gian).
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)  # Lần chạy 2.
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)  # Lần chạy 3.

In [ ]:
compile_and_run()  # Biên dịch và chạy bản C++ hiện có trong main.cpp.

In [ ]:
19.178207/0.082168  # Tính nhanh speedup (tăng tốc): thời gian Python / thời gian C++ của GPT-5 trên máy Ed.

## Được rồi, thử các model còn lại!

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)  # Claude viết C++, ghi đè main.cpp.
compile_and_run()  # Biên dịch và chạy bản Claude.

In [ ]:
port(grok, GROK_MODEL, pi)  # Grok viết C++, ghi đè main.cpp.
compile_and_run()  # Biên dịch và chạy bản Grok.

In [ ]:
port(gemini, GEMINI_MODEL, pi)  # Gemini viết C++, ghi đè main.cpp.
compile_and_run()  # Biên dịch và chạy bản Gemini.


In [ ]:
print(f"""
Trong thí nghiệm của Ed, các mức tăng tốc (performance speedup) là:

Hạng 4: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup (tăng tốc)
Hạng 3: GPT-5: {19.178207/0.082168:.0f}X speedup (tăng tốc)
Hạng 2: Grok 4: {19.178207/0.018092:.0f}X speedup (tăng tốc)
Hạng 1: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup (tăng tốc)
""")
# Các số 19.178207 / 0.104241 / ... là thời gian Python và C++ Ed đo được; {{:.0f}} làm tròn số lần tăng tốc.